<a href="https://colab.research.google.com/github/esneydersandoval31-alt/608142-DEEP-LEARNING--CONCEPTOS/blob/main/implementaci%C3%B3n_b%C3%A1sica_de_GANs_o_DCGANs_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PARTE 1: IMPLEMENTACIÓN DE DCGAN EN PYTORCH
# ==============================================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt

# 1. Configuración de dispositivo e hiperparámetros
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo activo: {device}")

batch_size = 128
image_size = 64
nc = 3          # Canales de color (RGB)
nz = 100        # Dimensión del vector latente z
ngf = 64        # Mapas de características en el Generador
ndf = 64        # Mapas de características en el Discriminador
num_epochs = 15 # Ajustable según tiempo disponible en Colab
lr = 0.0002
beta1 = 0.5

# 2. Preparación de datos (CIFAR-10 redimensionado a 64x64 para estabilidad)
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, shuffle=True, num_workers=2
)

# Inicialización de pesos según el paper original de DCGAN
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# 3. Arquitectura del Generador
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # Entrada: Z de tamaño (nz, 1, 1)
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # Estado: (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # Estado: (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # Estado: (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # Estado: (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # Salida: (nc) x 64 x 64
        )

    def forward(self, input):
        return self.main(input)

# 4. Arquitectura del Discriminador
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # Entrada: (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Estado: (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Estado: (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Estado: (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # Estado: (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

netG = Generator().to(device)
netG.apply(weights_init)

netD = Discriminator().to(device)
netD.apply(weights_init)

criterion = nn.BCELoss()
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# 5. Bucle de entrenamiento adversarial
G_losses, D_losses, img_list = [], [], []

print("Iniciando entrenamiento de DCGAN...")
for epoch in range(num_epochs):
    for i, data in enumerate(dataloader, 0):
        # (1) Actualizar Discriminador: maximizar log(D(x)) + log(1 - D(G(z)))
        netD.zero_grad()
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), 1.0, dtype=torch.float, device=device)

        output = netD(real_cpu).view(-1)
        errD_real = criterion(output, label)
        errD_real.backward()
        D_x = output.mean().item()

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.0)
        output = netD(fake.detach()).view(-1)
        errD_fake = criterion(output, label)
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        errD = errD_real + errD_fake
        optimizerD.step()

        # (2) Actualizar Generador: maximizar log(D(G(z)))
        netG.zero_grad()
        label.fill_(1.0)
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()
        optimizerG.step()

        G_losses.append(errG.item())
        D_losses.append(errD.item())

    print(f"Época [{epoch+1}/{num_epochs}] | Loss_D: {errD.item():.4f} | Loss_G: {errG.item():.4f} | D(x): {D_x:.4f} | D(G(z)): {D_G_z1:.4f}/{D_G_z2:.4f}")
    with torch.no_grad():
        fake_grid = netG(fixed_noise).detach().cpu()
    img_list.append(vutils.make_grid(fake_grid, padding=2, normalize=True))

# 6. Visualización de curvas de pérdida e imágenes generadas
plt.figure(figsize=(10, 4))
plt.title("Pérdidas del Generador y Discriminador durante el entrenamiento")
plt.plot(G_losses, label="Generador (G)")
plt.plot(D_losses, label="Discriminador (D)")
plt.xlabel("Iteraciones")
plt.ylabel("Pérdida BCE")
plt.legend()
plt.show()

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Imágenes sintéticas generadas (Última época)")
plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))
plt.show()

Dispositivo activo: cpu


 17%|█▋        | 29.8M/170M [02:11<10:21, 226kB/s]


KeyboardInterrupt: 